# Prompt Engineering to Agentic RAG 🤖🧠

## LangChain RAG Agent with AG News

### Shared retrieval utilities

These helpers are defined before CRAG because both the corrective retriever and the later comparison functions depend on them.

In [ ]:
from langchain_core.documents import Document

def document_identity(doc: Document) -> str:
    """
    Return a stable identifier for a retrieved document.

    Preference order:
    1. source metadata
    2. record_id metadata
    3. LangChain document id
    4. Python object identity as a last-resort fallback
    """
    return str(
        doc.metadata.get("source")
        or doc.metadata.get("record_id")
        or getattr(doc, "id", None)
        or id(doc)
    )


def merge_unique_documents(
    *document_lists: list[Document],
) -> list[Document]:
    """
    Merge retrieval results while preserving order and removing duplicates.
    """
    merged = []
    seen = set()

    for docs in document_lists:
        for doc in docs:
            identity = document_identity(doc)

            if identity not in seen:
                seen.add(identity)
                merged.append(doc)

    return merged


This notebook builds a Retrieval-Augmented Generation agent with LangChain and the public **AG News** dataset from Hugging Face.

It demonstrates:

- prompt engineering;
- context optimization based on token count;
- query rewriting;
- query expansion;
- step-back prompting;
- HyDE;
- self-querying retrieval with metadata filtering;
- Corrective RAG (CRAG) with relevance grading;
- baseline versus transformed retrieval.

> AG News is historical. Treat it as a news archive, not a live news service.

The shared retrieval utilities are defined immediately before this section so the CRAG grader and merge logic can use `document_identity()` and `merge_unique_documents()` without forward-reference errors.

## Learning objectives
9. Use a **Python logger** to record and monitor CRAG retrieval grades, relevance scores, correction decisions, latency, and token usage.

1. Load and index a public Hugging Face dataset.
2. Build a LangChain retrieval tool and RAG agent.
3. Optimize context with truncation, extractive compression, and conditional summarization.
4. Apply query rewriting, expansion, step-back prompting, and HyDE.
5. Implement self-querying retrieval with semantic search and metadata filtering.
6. Merge and compare retrieval results.
7. Implement Corrective RAG with conditional query correction.
8. Compare baseline, transformed, self-querying, and corrective RAG pipelines.

## 1. Install dependencies

In [ ]:
%pip install -U datasets langchain langchain-openai langchain-community langchain-text-splitters python-dotenv colorama tiktoken pandas

## 2. Environment setup

In [ ]:
import os
from getpass import getpass

from colorama import Fore, Style
from dotenv import load_dotenv

load_dotenv(override=True)

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")
else:
    print(f"{Fore.GREEN}OpenAI API key loaded.{Style.RESET_ALL}")

## CRAG Observability with a Python Logger

The CRAG pipeline already contains an LLM-based retrieval grader through
`DocumentRelevanceGrade`.

That class answers a semantic evaluation question:

> **Is this retrieved document relevant to the user's question?**

The Python logger does not change that decision. It simply records the results
so we can monitor retrieval behavior across runs.

```text
Question
   ↓
Retrieve documents
   ↓
DocumentRelevanceGrade
   ↓
relevant / not_relevant + reason
   ↓
Python Logger
   ├── retrieval path
   ├── relevance rate
   ├── correction decision
   ├── document grades
   ├── token usage
   └── operational metrics
```

This notebook keeps the existing RAG and CRAG implementation unchanged and adds
only a lightweight logging layer around the retrieval process.


In [ ]:
import logging
import sys

from colorama import Fore, Style


crag_logger = logging.getLogger("crag_retrieval")
crag_logger.setLevel(logging.INFO)
crag_logger.propagate = False

if not crag_logger.handlers:
    console_handler = logging.StreamHandler(sys.stdout)
    console_handler.setFormatter(
        logging.Formatter(
            "%(asctime)s | %(levelname)-8s | %(message)s",
            datefmt="%H:%M:%S",
        )
    )
    crag_logger.addHandler(console_handler)

print(
    f"{Fore.CYAN}{Style.BRIGHT}"
    "CRAG retrieval logger initialized."
    f"{Style.RESET_ALL}"
)


## 3. Initialize the model

In [ ]:
from langchain.chat_models import init_chat_model

model = init_chat_model(
    "openai:gpt-4.1-mini",
    temperature=0,
    timeout=30,
    max_retries=3,
)

print(f"{Fore.CYAN}Model initialized.{Style.RESET_ALL}")

## 4. Define prompts

In [ ]:
WEAK_PROMPT = """You are a news assistant.
Use the available search tool and answer the user's question."""

GROUNDED_PROMPT = """You are a grounded news-archive assistant.

Use the available search tool before answering questions about the archive.

Rules:
- Use only retrieved evidence.
- Cite factual claims with [Source N].
- Never present historical archive data as current.
- If evidence is insufficient, say so.
- Begin with a direct answer.
- Add up to three concise cited bullets.
- Finish with one limitation.
- Use no more than 180 words.
"""

BEGINNER_FRIENDLY_PROMPT = """You are a teacher explaining historical news to beginners.

Use the available search tool and only retrieved evidence.
Explain difficult terms, use at most three bullets, include one example,
cite factual claims with [Source N], and end with 'In simple terms, ...'.
Write 140 to 180 words.
"""

EXECUTIVE_PROMPT = """You are preparing a briefing for a business leader.

Use the available search tool and only retrieved evidence.

Return exactly:
## Headline
## Key Signals
## Business Relevance
## Data Limitation

Use direct language and no more than 140 words.
"""

CLARIFYING_PROMPT = """You are a grounded news-archive assistant.

If the request is ambiguous, ask one concise clarification question and do not search yet.
Otherwise use the available search tool, cite [Source N], and state when evidence is insufficient.
"""

ANALYTICAL_PROMPT = """You are an analytical research assistant.

Use the available search tool and only retrieved evidence.

Return exactly:
## Main Finding
## Evidence
## Interpretation
## Limitation

Use neutral language and no more than 220 words.
"""

## 5. Load the AG News dataset

In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "fancyzhx/ag_news",
    split="train[:500]",
)

print(dataset)
print(dataset[0])

## 6. Convert rows into LangChain documents

In [ ]:
from langchain_core.documents import Document

LABELS = {
    0: "World",
    1: "Sports",
    2: "Business",
    3: "Science and Technology",
}

documents = []

for index, row in enumerate(dataset):
    label = int(row["label"])
    documents.append(
        Document(
            page_content=row["text"].strip(),
            metadata={
                "source": f"ag_news_{index}",
                "category": LABELS[label],
                "dataset": "AG News",
                "record_id": index,
            },
        )
    )

print(f"Documents created: {len(documents)}")
print(documents[0])

## 7. Split, embed, and index

In [ ]:
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
)

chunks = splitter.split_documents(documents)

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
)

vector_store = InMemoryVectorStore(
    embedding=embeddings,
)

vector_store.add_documents(chunks)

baseline_retriever = vector_store.as_retriever(
    search_kwargs={"k": 5},
)

broad_retriever = vector_store.as_retriever(
    search_kwargs={"k": 12},
)

print(f"Documents: {len(documents)}")
print(f"Chunks: {len(chunks)}")

## 8. Formatting and token utilities

In [ ]:
import tiktoken

TOKENIZER = tiktoken.encoding_for_model("gpt-4.1-mini")


def count_tokens(text: str) -> int:
    return len(TOKENIZER.encode(text))


def document_tokens(doc: Document) -> int:
    return count_tokens(doc.page_content)


def format_documents(docs: list[Document]) -> str:
    if not docs:
        return "No relevant articles were retrieved."

    parts = []

    for position, doc in enumerate(docs, start=1):
        parts.append(
            f"[Source {position}]\n"
            f"Record: {doc.metadata.get('source', 'unknown')}\n"
            f"Category: {doc.metadata.get('category', 'unknown')}\n"
            f"Article: {doc.page_content}"
        )

    return "\n\n".join(parts)


def inspect_token_usage(
    docs: list[Document],
    label: str = "Context",
) -> dict:
    counts = [document_tokens(doc) for doc in docs]

    stats = {
        "label": label,
        "documents": len(docs),
        "total_tokens": sum(counts),
        "average_tokens": round(sum(counts) / len(counts), 1) if counts else 0,
        "largest_document": max(counts, default=0),
    }

    print(stats)
    return stats

## 9. Context optimization

The optimizer runs automatically from token counts:

1. Keep full context when it fits.
2. Compress relevant sentences when it is too large.
3. Truncate to the token budget.
4. Summarize only if the result remains above the preferred threshold.

In [ ]:
import re
from math import sqrt

from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate


def truncate_documents_by_tokens(
    docs: list[Document],
    max_tokens: int = 900,
) -> tuple[list[Document], dict]:
    selected = []
    excluded = []
    used_tokens = 0

    for doc in docs:
        tokens = document_tokens(doc)

        if used_tokens + tokens <= max_tokens:
            selected.append(doc)
            used_tokens += tokens
        else:
            excluded.append(doc)

    return selected, {
        "budget": max_tokens,
        "used_tokens": used_tokens,
        "selected_documents": len(selected),
        "excluded_documents": len(excluded),
    }


def cosine_similarity(
    vector_a: list[float],
    vector_b: list[float],
) -> float:
    numerator = sum(a * b for a, b in zip(vector_a, vector_b))
    norm_a = sqrt(sum(a * a for a in vector_a))
    norm_b = sqrt(sum(b * b for b in vector_b))

    if not norm_a or not norm_b:
        return 0.0

    return numerator / (norm_a * norm_b)


def split_sentences(text: str) -> list[str]:
    return [
        sentence.strip()
        for sentence in re.split(r"(?<=[.!?])\s+", text)
        if sentence.strip()
    ]


def compress_documents(
    query: str,
    docs: list[Document],
    threshold: float = 0.20,
    max_sentences_per_doc: int = 2,
) -> list[Document]:
    if not docs:
        return []

    query_vector = embeddings.embed_query(query)
    document_sentences = [
        split_sentences(doc.page_content)
        for doc in docs
    ]
    all_sentences = [
        sentence
        for sentences in document_sentences
        for sentence in sentences
    ]

    if not all_sentences:
        return []

    all_vectors = embeddings.embed_documents(all_sentences)

    output = []
    vector_index = 0

    for doc, sentences in zip(docs, document_sentences):
        sentence_count = len(sentences)
        vectors = all_vectors[
            vector_index: vector_index + sentence_count
        ]
        vector_index += sentence_count

        scored_sentences = sorted(
            [
                (
                    cosine_similarity(query_vector, vector),
                    sentence,
                )
                for sentence, vector in zip(sentences, vectors)
            ],
            key=lambda item: item[0],
            reverse=True,
        )

        kept = [
            sentence
            for score, sentence in scored_sentences[:max_sentences_per_doc]
            if score >= threshold
        ]

        if kept:
            output.append(
                Document(
                    page_content=" ".join(kept),
                    metadata={
                        **doc.metadata,
                        "compression": "extractive",
                    },
                )
            )

    return output


SUMMARY_PROMPT = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """Compress retrieved news evidence for a downstream RAG answer.

Preserve:
- facts needed to answer the question;
- source labels exactly as [Source N];
- separate events;
- uncertainty and limitations.

Do not add outside information.
Return no more than {target_words} words.""",
        ),
        (
            "human",
            """Question:
{question}

Retrieved evidence:
{context}

Compressed evidence:""",
        ),
    ]
)

summary_chain = SUMMARY_PROMPT | model | StrOutputParser()


def summarize_context(
    question: str,
    docs: list[Document],
    target_words: int = 180,
) -> str:
    return summary_chain.invoke(
        {
            "question": question,
            "context": format_documents(docs),
            "target_words": target_words,
        }
    )


def optimize_context(
    query: str,
    docs: list[Document],
    max_context_tokens: int = 900,
    summarize_threshold: int = 700,
) -> tuple[str, dict]:
    before_context = format_documents(docs)
    before_tokens = count_tokens(before_context)

    details = {
        "action": "full_context",
        "documents_before": len(docs),
    }

    if before_tokens <= max_context_tokens:
        context = before_context

    else:
        compressed_docs = compress_documents(
            query=query,
            docs=docs,
        )

        if not compressed_docs:
            compressed_docs = docs
            details["compression_fallback"] = True

        details["documents_after_compression"] = len(compressed_docs)

        selected_docs, truncation_report = truncate_documents_by_tokens(
            docs=compressed_docs,
            max_tokens=max_context_tokens,
        )

        context = format_documents(selected_docs)
        details.update(truncation_report)
        details["action"] = "compress_then_truncate"

        if selected_docs and count_tokens(context) > summarize_threshold:
            context = summarize_context(
                question=query,
                docs=selected_docs,
                target_words=180,
            )
            details["action"] = "compress_truncate_then_summarize"

    after_tokens = count_tokens(context)

    return context, {
        "before_tokens": before_tokens,
        "after_tokens": after_tokens,
        "token_reduction": before_tokens - after_tokens,
        **details,
    }

### Context optimization demo

In [ ]:
optimization_query = "technology companies and business risks"
optimization_docs = broad_retriever.invoke(optimization_query)

optimized_context, optimization_report = optimize_context(
    query=optimization_query,
    docs=optimization_docs,
)

print(optimization_report)
print("\nOptimized context:\n")
print(optimized_context[:1500])

# Query Transformation

The raw user question may be vague or use different vocabulary from the archive.

This section adds four complementary techniques:

1. **Query rewriting** — creates one clearer retrieval query.
2. **Query expansion** — generates alternative phrasings.
3. **Step-back prompting** — retrieves broader conceptual context.
4. **HyDE** — retrieves using an embedding of a hypothetical answer.

## 10. Query rewriting

In [ ]:
REWRITE_PROMPT = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """Rewrite the user's question as one clear, keyword-rich search query
for a historical news archive.

Do not answer the question.
Return only the rewritten query.""",
        ),
        ("human", "{question}"),
    ]
)

rewrite_chain = REWRITE_PROMPT | model | StrOutputParser()


def rewrite_query(question: str) -> str:
    return rewrite_chain.invoke(
        {"question": question}
    ).strip()


example_question = "what's going on with tech stocks lately"
rewritten = rewrite_query(example_question)

print(f"Original: {example_question}")
print(f"Rewritten: {rewritten}")

## 11. Query expansion

In [ ]:
EXPANSION_PROMPT = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """Generate exactly {n} retrieval queries that explore different
phrasings, synonyms, or angles of the user's question.

Return one query per line.
Do not use numbering or add explanations.""",
        ),
        ("human", "{question}"),
    ]
)

expansion_chain = EXPANSION_PROMPT | model | StrOutputParser()


def expand_query(
    question: str,
    n: int = 3,
) -> list[str]:
    raw = expansion_chain.invoke(
        {
            "question": question,
            "n": n,
        }
    )

    queries = [
        line.strip("-• ").strip()
        for line in raw.splitlines()
        if line.strip()
    ]

    return queries[:n]


def retrieve_with_expansion(
    question: str,
    n: int = 3,
    k: int = 5,
) -> list[Document]:
    queries = [question, *expand_query(question, n=n)]

    merged = []
    seen_sources = set()

    for query in queries:
        for doc in baseline_retriever.invoke(query):
            source = doc.metadata.get("source", doc.id)

            if source not in seen_sources:
                seen_sources.add(source)
                merged.append(doc)

    return merged[:k]


example_question = "technology companies and business risks"
expanded_queries = expand_query(example_question)

print(expanded_queries)
inspect_token_usage(
    retrieve_with_expansion(example_question),
    "Expansion retrieval",
)

## 12. Step-back prompting

In [ ]:
STEP_BACK_PROMPT = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """Rewrite the user's question as one broader, higher-level question
that captures the underlying concepts.

Do not answer the question.
Return only the broader question.""",
        ),
        ("human", "{question}"),
    ]
)

step_back_chain = STEP_BACK_PROMPT | model | StrOutputParser()


def step_back_query(question: str) -> str:
    return step_back_chain.invoke(
        {"question": question}
    ).strip()


def retrieve_with_step_back(
    question: str,
    k: int = 5,
) -> tuple[list[Document], str]:
    broader_question = step_back_query(question)
    documents = baseline_retriever.invoke(broader_question)[:k]
    return documents, broader_question


example_question = "What business risks do technology companies face?"
step_back_docs, broader_question = retrieve_with_step_back(
    example_question
)

print(f"Original: {example_question}")
print(f"Step-back: {broader_question}")
inspect_token_usage(step_back_docs, "Step-back retrieval")

## 13. HyDE

In [ ]:
HYDE_PROMPT = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """Write a short hypothetical historical-news passage that could
answer the user's question.

Do not say it is hypothetical.
Use two or three sentences.""",
        ),
        ("human", "{question}"),
    ]
)

hyde_chain = HYDE_PROMPT | model | StrOutputParser()


def hyde_retrieve(
    question: str,
    k: int = 5,
) -> tuple[list[Document], str]:
    hypothetical_document = hyde_chain.invoke(
        {"question": question}
    ).strip()

    hyde_vector = embeddings.embed_query(
        hypothetical_document
    )

    documents = vector_store.similarity_search_by_vector(
        hyde_vector,
        k=k,
    )

    return documents, hypothetical_document


example_question = "technology companies and business risks"
hyde_docs, hypothetical_document = hyde_retrieve(
    example_question
)

print("Hypothetical document:")
print(hypothetical_document)
inspect_token_usage(hyde_docs, "HyDE retrieval")

## 14. Self-querying retrieval

A self-querying retriever converts a natural-language request into:

1. a **semantic query** describing what the user wants to find;
2. a **metadata filter** narrowing the search space.

The AG News documents already contain a `category` metadata field:

- `World`
- `Sports`
- `Business`
- `Science and Technology`

For example:

```text
User request:
Show me business articles about technology-company risks.

Structured search:
semantic_query = "technology-company risks"
category = "Business"
```

The vector store then performs semantic search only within the matching category.
This improves precision when the user's request contains both a topic and a
structured constraint.


In [ ]:
from typing import Literal, Optional

from pydantic import BaseModel, Field


NewsCategory = Literal[
    "World",
    "Sports",
    "Business",
    "Science and Technology",
]


class StructuredNewsQuery(BaseModel):
    """Semantic query and optional AG News category filter."""

    semantic_query: str = Field(
        description=(
            "A concise semantic search query containing only the topic, "
            "entities, or event to retrieve."
        )
    )
    category: Optional[NewsCategory] = Field(
        default=None,
        description=(
            "The AG News category explicitly requested or strongly implied "
            "by the user. Use null when no category constraint is present."
        ),
    )


SELF_QUERY_PROMPT = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """Convert the user's request into a semantic search query and an
optional AG News category filter.

Available categories:
- World
- Sports
- Business
- Science and Technology

Rules:
- Remove category words from semantic_query when they are represented by the filter.
- Do not invent a category.
- Use category=null when no category is requested or clearly implied.
- Do not answer the user's question.""",
        ),
        ("human", "{question}"),
    ]
)

self_query_chain = (
    SELF_QUERY_PROMPT
    | model.with_structured_output(StructuredNewsQuery)
)


def self_query_retrieve(
    question: str,
    k: int = 5,
) -> tuple[list[Document], StructuredNewsQuery]:
    """Run semantic search with an LLM-generated metadata category filter."""

    structured_query = self_query_chain.invoke(
        {"question": question}
    )

    def category_filter(doc: Document) -> bool:
        if structured_query.category is None:
            return True

        return (
            doc.metadata.get("category")
            == structured_query.category
        )

    try:
        # InMemoryVectorStore accepts a callable metadata/document filter.
        docs = vector_store.similarity_search(
            query=structured_query.semantic_query,
            k=k,
            filter=category_filter,
        )
    except (TypeError, NotImplementedError):
        # Compatibility fallback for vector stores that cannot pre-filter.
        candidates = vector_store.similarity_search(
            query=structured_query.semantic_query,
            k=max(k * 4, 20),
        )
        docs = [
            doc
            for doc in candidates
            if category_filter(doc)
        ][:k]

    return docs, structured_query


self_query_example = (
    "Show me Business articles about technology-company risks."
)

self_query_docs, structured_query = self_query_retrieve(
    self_query_example
)

print(f"{Fore.CYAN}Original request:{Fore.RESET}")
print(self_query_example)

print(f"\n{Fore.CYAN}Structured query:{Fore.RESET}")
print(structured_query.model_dump())

inspect_token_usage(
    self_query_docs,
    "Self-query retrieval",
)

print(f"\n{Fore.GREEN}Retrieved categories:{Fore.RESET}")
print([
    doc.metadata.get("category")
    for doc in self_query_docs
])

## 15. Corrective RAG (CRAG)

**Corrective Retrieval-Augmented Generation (CRAG)** adds a relevance-checking
step between retrieval and answer generation.

A standard RAG flow is:

```text
Question → Retrieve → Generate
```

CRAG checks whether the retrieved evidence is actually useful:

```text
Question
   ↓
Initial retrieval
   ↓
Document relevance grading
   ↓
Useful evidence? ── yes ──→ keep relevant documents
       │
       no
       ↓
Rewrite the query and retrieve again
   ↓
Merge corrected evidence
   ↓
Generate a grounded answer
```

This implementation reuses the notebook's existing LangChain model,
retrievers, query-rewriting chain, document utilities, and context optimizer.
The corrective branch runs only when the initial retrieval produces fewer than
the required number of relevant documents.

CRAG can improve robustness, but it adds latency and model calls because each
candidate document is graded before the final context is assembled.


In [ ]:
class DocumentRelevanceGrade(BaseModel):
    """Structured CRAG relevance assessment for one document."""

    relevance: Literal["relevant", "not_relevant"] = Field(
        description=(
            "Whether the document contains evidence that could help "
            "answer the user's question."
        )
    )
    reason: str = Field(
        description="A concise explanation of the relevance decision."
    )


CRAG_GRADER_PROMPT = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """You grade evidence retrieved from a historical news archive.

Decide whether the supplied document contains information that could help
answer the user's question.

Rules:
- Judge only the supplied document.
- Accept useful facts, entities, events, causes, consequences, or background.
- Do not require an exact keyword match.
- Reject unrelated or merely superficial matches.
- Keep the reason concise.""",
        ),
        (
            "human",
            """Question:
{question}

Retrieved document:
{document}""",
        ),
    ]
)

crag_grader_chain = (
    CRAG_GRADER_PROMPT
    | model.with_structured_output(DocumentRelevanceGrade)
)


def grade_retrieved_documents(
    question: str,
    docs: list[Document],
) -> tuple[list[Document], list[dict]]:
    """Grade documents and return the relevant subset plus diagnostics."""
    relevant_docs = []
    grading_report = []

    for doc in docs:
        grade = crag_grader_chain.invoke(
            {
                "question": question,
                "document": doc.page_content,
            }
        )

        grading_report.append(
            {
                "source": document_identity(doc),
                "category": doc.metadata.get("category", "unknown"),
                "relevance": grade.relevance,
                "reason": grade.reason,
            }
        )

        if grade.relevance == "relevant":
            relevant_docs.append(doc)

    return relevant_docs, grading_report


In [ ]:
def crag_retrieve(
    question: str,
    k: int = 5,
    minimum_relevant_docs: int = 2,
) -> tuple[list[Document], dict]:
    """
    Retrieve, evaluate, and conditionally correct weak retrieval results.
    """
    initial_docs = baseline_retriever.invoke(question)[:k]

    relevant_initial_docs, initial_grades = grade_retrieved_documents(
        question=question,
        docs=initial_docs,
    )

    initial_relevance_rate = (
        len(relevant_initial_docs) / len(initial_docs)
        if initial_docs
        else 0.0
    )

    correction_required = (
        len(relevant_initial_docs) < minimum_relevant_docs
    )

    corrected_query = None
    corrective_candidates = []
    relevant_corrective_docs = []
    corrective_grades = []

    if correction_required:
        corrected_query = rewrite_query(question)
        corrective_candidates = broad_retriever.invoke(corrected_query)

        relevant_corrective_docs, corrective_grades = (
            grade_retrieved_documents(
                question=question,
                docs=corrective_candidates,
            )
        )

    final_docs = merge_unique_documents(
        relevant_initial_docs,
        relevant_corrective_docs,
    )[:k]

    # Preserve a limited fallback so the downstream agent can explicitly
    # report insufficient evidence instead of receiving an empty context.
    used_fallback = False
    if not final_docs:
        final_docs = initial_docs[:k]
        used_fallback = True

    report = {
        "retrieval_path": (
            "corrective_retrieval"
            if correction_required
            else "initial_retrieval_accepted"
        ),
        "initial_documents": len(initial_docs),
        "initial_relevant_documents": len(relevant_initial_docs),
        "initial_relevance_rate": round(initial_relevance_rate, 3),
        "minimum_relevant_documents": minimum_relevant_docs,
        "correction_required": correction_required,
        "corrected_query": corrected_query,
        "corrective_candidates": len(corrective_candidates),
        "corrective_relevant_documents": len(
            relevant_corrective_docs
        ),
        "final_documents": len(final_docs),
        "used_fallback": used_fallback,
        "initial_grades": initial_grades,
        "corrective_grades": corrective_grades,
        "sources": [
            document_identity(doc)
            for doc in final_docs
        ],
    }

    return final_docs, report


In [ ]:
crag_question = (
    "What business risks do technology companies face?"
)

crag_docs, crag_report = crag_retrieve(
    question=crag_question,
    k=5,
    minimum_relevant_docs=2,
)

print(
    f"{Fore.CYAN}{Style.BRIGHT}"
    "=== CRAG Retrieval Report ==="
    f"{Style.RESET_ALL}"
)
print(f"Question: {crag_question}")
print(f"Retrieval path: {crag_report['retrieval_path']}")
print(
    "Initial relevant documents: "
    f"{crag_report['initial_relevant_documents']}/"
    f"{crag_report['initial_documents']}"
)
print(
    "Initial relevance rate: "
    f"{crag_report['initial_relevance_rate']:.0%}"
)
print(
    "Correction required: "
    f"{crag_report['correction_required']}"
)
print(
    "Corrected query: "
    f"{crag_report['corrected_query'] or 'Not required'}"
)
print(f"Final documents: {crag_report['final_documents']}")

print(
    f"\n{Fore.GREEN}{Style.BRIGHT}"
    "Initial document grades:"
    f"{Style.RESET_ALL}"
)

for grade in crag_report["initial_grades"]:
    grade_color = (
        Fore.GREEN
        if grade["relevance"] == "relevant"
        else Fore.RED
    )
    print(
        f"{grade_color}"
        f"{grade['source']:<15} "
        f"{grade['relevance']:<13}"
        f"{Style.RESET_ALL} "
        f"{grade['reason']}"
    )

if crag_report["corrective_grades"]:
    print(
        f"\n{Fore.GREEN}{Style.BRIGHT}"
        "Corrective document grades:"
        f"{Style.RESET_ALL}"
    )

    for grade in crag_report["corrective_grades"]:
        grade_color = (
            Fore.GREEN
            if grade["relevance"] == "relevant"
            else Fore.RED
        )
        print(
            f"{grade_color}"
            f"{grade['source']:<15} "
            f"{grade['relevance']:<13}"
            f"{Style.RESET_ALL} "
            f"{grade['reason']}"
        )

inspect_token_usage(crag_docs, "CRAG retrieval")


### Log CRAG Retrieval Results

The existing `crag_report` already contains the information we need:

- number of initial documents;
- number of relevant documents;
- initial relevance rate;
- whether corrective retrieval was required;
- corrected query;
- final document count;
- per-document relevance grades and reasons.

The logger records those values without changing the CRAG pipeline.

For readability:

- normal/relevant results are logged as `INFO`;
- rejected documents and corrective retrieval are logged as `WARNING`;
- Colorama is used in the notebook output to make the summary easier to scan.


In [ ]:
print(
    f"\n{Fore.CYAN}{Style.BRIGHT}"
    "=== CRAG LOGGER SUMMARY ==="
    f"{Style.RESET_ALL}"
)

print(
    f"{Fore.YELLOW}Question:{Style.RESET_ALL} "
    f"{crag_question}"
)

relevance_color = (
    Fore.GREEN
    if crag_report["initial_relevance_rate"] >= 0.8
    else Fore.YELLOW
    if crag_report["initial_relevance_rate"] >= 0.6
    else Fore.RED
)

print(
    "Initial relevance rate: "
    f"{relevance_color}"
    f"{crag_report['initial_relevance_rate']:.0%}"
    f"{Style.RESET_ALL}"
)

print(
    "Correction required: "
    f"{Fore.RED if crag_report['correction_required'] else Fore.GREEN}"
    f"{crag_report['correction_required']}"
    f"{Style.RESET_ALL}"
)

crag_logger.info(
    "question=%r | path=%s | initial_relevant=%s/%s | "
    "relevance_rate=%s | correction_required=%s | "
    "corrected_query=%r | final_documents=%s",
    crag_question,
    crag_report["retrieval_path"],
    crag_report["initial_relevant_documents"],
    crag_report["initial_documents"],
    f"{crag_report['initial_relevance_rate']:.0%}",
    crag_report["correction_required"],
    crag_report["corrected_query"],
    crag_report["final_documents"],
)

if crag_report["correction_required"]:
    crag_logger.warning(
        "corrective retrieval triggered | corrected_query=%r | "
        "candidates=%s | relevant_corrective_documents=%s",
        crag_report["corrected_query"],
        crag_report["corrective_candidates"],
        crag_report["corrective_relevant_documents"],
    )


### Log `DocumentRelevanceGrade` Decisions

Every retrieved document already receives a structured grade from the CRAG
evaluator.

The logger records:

- source;
- category;
- relevance decision;
- reason.

This provides a persistent-style execution record of the same grades that are
printed interactively in the CRAG report.


In [ ]:
print(
    f"\n{Fore.MAGENTA}{Style.BRIGHT}"
    "=== LOGGED DOCUMENT RELEVANCE GRADES ==="
    f"{Style.RESET_ALL}"
)

for grade in crag_report["initial_grades"]:
    grade_color = (
        Fore.GREEN
        if grade["relevance"] == "relevant"
        else Fore.RED
    )

    print(
        f"{grade_color}"
        f"{grade['source']:<16}"
        f"{grade['relevance']:<14}"
        f"{Style.RESET_ALL}"
        f"{grade['reason']}"
    )

    log_method = (
        crag_logger.info
        if grade["relevance"] == "relevant"
        else crag_logger.warning
    )

    log_method(
        "initial grade | source=%s | category=%s | "
        "relevance=%s | reason=%s",
        grade["source"],
        grade["category"],
        grade["relevance"],
        grade["reason"],
    )

for grade in crag_report["corrective_grades"]:
    grade_color = (
        Fore.GREEN
        if grade["relevance"] == "relevant"
        else Fore.RED
    )

    print(
        f"{grade_color}"
        f"{grade['source']:<16}"
        f"{grade['relevance']:<14}"
        f"{Style.RESET_ALL}"
        f"{grade['reason']}"
    )

    log_method = (
        crag_logger.info
        if grade["relevance"] == "relevant"
        else crag_logger.warning
    )

    log_method(
        "corrective grade | source=%s | category=%s | "
        "relevance=%s | reason=%s",
        grade["source"],
        grade["category"],
        grade["relevance"],
        grade["reason"],
    )


### Record Token Usage

Retrieval quality should be monitored together with efficiency.

The existing `inspect_token_usage()` helper already measures the size of the
retrieved context. We simply log the returned values.


In [ ]:
crag_token_report = inspect_token_usage(
    crag_docs,
    "CRAG retrieval",
)

crag_logger.info(
    "token usage | documents=%s | total_tokens=%s | "
    "average_tokens=%s | largest_document=%s",
    crag_token_report["documents"],
    crag_token_report["total_tokens"],
    crag_token_report["average_tokens"],
    crag_token_report["largest_document"],
)


### Conclusion

`DocumentRelevanceGrade` remains the **LLM-based retrieval evaluator**.

The Python logger adds observability around that existing process:

- the grader decides whether a document is relevant;
- CRAG decides whether corrective retrieval is necessary;
- the logger records those decisions, scores, reasons, and efficiency signals.

This gives us a repeatable way to **measure and monitor retrieval behavior**
without modifying the retrieval logic itself.


## 16. Retrieval with Query Transformation and self-querying

In [ ]:
def retrieve_with_all_transformations(
    question: str,
    k: int = 8,
) -> tuple[list[Document], dict]:
    rewritten_query = rewrite_query(question)
    expanded_queries = expand_query(question)

    baseline_docs = baseline_retriever.invoke(question)
    rewritten_docs = baseline_retriever.invoke(rewritten_query)
    expanded_docs = retrieve_with_expansion(
        question=question,
        k=k,
    )
    step_back_docs, broader_question = retrieve_with_step_back(
        question=question,
        k=k,
    )
    hyde_docs, hypothetical_document = hyde_retrieve(
        question=question,
        k=k,
    )

    merged_docs = merge_unique_documents(
        baseline_docs,
        rewritten_docs,
        expanded_docs,
        step_back_docs,
        hyde_docs,
    )[:k]

    details = {
        "rewritten_query": rewritten_query,
        "expanded_queries": expanded_queries,
        "step_back_query": broader_question,
        "hypothetical_document": hypothetical_document,
        "baseline_documents": len(baseline_docs),
        "rewrite_documents": len(rewritten_docs),
        "expansion_documents": len(expanded_docs),
        "step_back_documents": len(step_back_docs),
        "hyde_documents": len(hyde_docs),
        "documents_retrieved": len(merged_docs),
        "sources": [
            document_identity(doc)
            for doc in merged_docs
        ],
    }

    return merged_docs, details

### Compare individual retrieval techniques

In [ ]:
import pandas as pd


def compare_retrieval_techniques(
    question: str,
    k: int = 5,
) -> pd.DataFrame:
    baseline_docs = baseline_retriever.invoke(question)
    rewritten_docs = baseline_retriever.invoke(
        rewrite_query(question)
    )
    expanded_docs = retrieve_with_expansion(
        question=question,
        k=k,
    )
    step_back_docs, _ = retrieve_with_step_back(
        question=question,
        k=k,
    )
    hyde_docs, _ = hyde_retrieve(
        question=question,
        k=k,
    )
    combined_docs, _ = retrieve_with_all_transformations(
        question=question,
        k=k,
    )

    baseline_sources = {
        document_identity(doc)
        for doc in baseline_docs
    }

    results = {
        "baseline": baseline_docs,
        "rewrite": rewritten_docs,
        "expansion": expanded_docs,
        "step_back": step_back_docs,
        "hyde": hyde_docs,
        "combined": combined_docs,
    }

    rows = []

    for technique, docs in results.items():
        sources = {
            document_identity(doc)
            for doc in docs
        }

        rows.append(
            {
                "technique": technique,
                "documents_retrieved": len(docs),
                "unique_sources": len(sources),
                "new_vs_baseline": len(
                    sources - baseline_sources
                ),
            }
        )

    return pd.DataFrame(rows)


comparison_question = "technology companies and business risks"
table = compare_retrieval_techniques(comparison_question)

## 17. Compare retrieval before and after transformation

This comparison measures three retrieval paths using the same natural-language request:

- **Baseline:** embeds the user request exactly as written.
- **Query transformed:** combines rewriting, expansion, step-back prompting, and HyDE.
- **Self-querying:** separates the semantic topic from the metadata category filter.

The `category_match_rate` metric shows how many returned documents satisfy the requested structured constraint. A higher rate indicates better metadata precision, but it does not by itself prove that the documents answer the question. Relevance should still be evaluated separately.

In [ ]:
import pandas as pd

from colorama import Fore, Style, init


# Initialize Colorama.
init(autoreset=False)


def category_match_rate(
    docs: list[Document],
    expected_category: str | None,
) -> float | None:
    """
    Return the proportion of retrieved documents whose category
    matches the expected category.

    Examples:
        5 matching documents out of 5 -> 1.0
        4 matching documents out of 5 -> 0.8
        No expected category          -> None
    """
    if expected_category is None:
        return None

    if not docs:
        return 0.0

    matches = sum(
        doc.metadata.get("category") == expected_category
        for doc in docs
    )

    return round(matches / len(docs), 3)


def print_colored_dataframe(
    dataframe: pd.DataFrame,
    title: str,
    color: str = Fore.CYAN,
) -> None:
    """
    Print the retrieval comparison as a terminal table.

    The original category_match_rate remains numeric in the DataFrame,
    while this table displays it as a colored percentage.
    """
    print(
        f"\n{color}{Style.BRIGHT}"
        f"{'=' * 110}\n"
        f"{title.center(110)}\n"
        f"{'=' * 110}"
        f"{Style.RESET_ALL}"
    )

    if dataframe.empty:
        print(
            f"{Fore.YELLOW}"
            "No results to display."
            f"{Style.RESET_ALL}"
        )
        return

    print(
        f"{Style.BRIGHT}"
        f"{'Technique':<16}"
        f"{'Docs':>8}"
        f"{'Sources':>10}"
        f"{'New':>8}"
        f"{'Category Match Rate':>24}"
        f"   {'Categories'}"
        f"{Style.RESET_ALL}"
    )

    print("-" * 110)

    for _, row in dataframe.iterrows():
        # row.get() prevents KeyError if the column is absent.
        rate = row.get("category_match_rate", None)

        if rate is None or pd.isna(rate):
            rate_text = "N/A"
            rate_color = Fore.WHITE
        else:
            percentage = float(rate) * 100
            rate_text = f"{percentage:.0f}%"

            if percentage >= 100:
                rate_color = Fore.GREEN
            elif percentage >= 80:
                rate_color = Fore.CYAN
            elif percentage >= 60:
                rate_color = Fore.YELLOW
            else:
                rate_color = Fore.RED

        print(
            f"{row.get('technique', 'unknown'):<16}"
            f"{row.get('documents_retrieved', 0):>8}"
            f"{row.get('unique_sources', 0):>10}"
            f"{row.get('new_vs_baseline', 0):>8}"
            f"{rate_color}"
            f"{rate_text:>24}"
            f"{Style.RESET_ALL}"
            f"   {row.get('categories', 'unknown')}"
        )


def compare_retrieval_techniques(
    question: str,
    expected_category: str | None = None,
    k: int = 5,
) -> pd.DataFrame:
    """
    Compare retrieval diversity and category precision across:

    - baseline retrieval
    - query rewriting
    - query expansion
    - step-back prompting
    - HyDE
    - combined query transformations
    - self-querying with metadata filtering
    """

    # ---------------------------------------------------------
    # 1. Baseline retrieval
    # ---------------------------------------------------------
    baseline_docs = baseline_retriever.invoke(question)[:k]

    # ---------------------------------------------------------
    # 2. Query rewriting
    # ---------------------------------------------------------
    rewritten_query = rewrite_query(question)

    rewritten_docs = baseline_retriever.invoke(
        rewritten_query
    )[:k]

    # ---------------------------------------------------------
    # 3. Query expansion
    # ---------------------------------------------------------
    expanded_docs = retrieve_with_expansion(
        question=question,
        k=k,
    )[:k]

    # ---------------------------------------------------------
    # 4. Step-back prompting
    # ---------------------------------------------------------
    step_back_docs, step_back_query_text = retrieve_with_step_back(
        question=question,
        k=k,
    )

    step_back_docs = step_back_docs[:k]

    # ---------------------------------------------------------
    # 5. HyDE
    # ---------------------------------------------------------
    hyde_docs, hypothetical_document = hyde_retrieve(
        question=question,
        k=k,
    )

    hyde_docs = hyde_docs[:k]

    # ---------------------------------------------------------
    # 6. Combined query transformations
    # ---------------------------------------------------------
    combined_docs, transformations = (
        retrieve_with_all_transformations(
            question=question,
            k=k,
        )
    )

    combined_docs = combined_docs[:k]

    # ---------------------------------------------------------
    # 7. Self-querying with metadata filtering
    # ---------------------------------------------------------
    self_query_docs, structured_query = self_query_retrieve(
        question=question,
        k=k,
    )

    self_query_docs = self_query_docs[:k]

    # ---------------------------------------------------------
    # Baseline source identities
    # ---------------------------------------------------------
    baseline_sources = {
        document_identity(doc)
        for doc in baseline_docs
    }

    # ---------------------------------------------------------
    # Group all retrieval results
    # ---------------------------------------------------------
    results = {
        "baseline": baseline_docs,
        "rewrite": rewritten_docs,
        "expansion": expanded_docs,
        "step_back": step_back_docs,
        "hyde": hyde_docs,
        "combined": combined_docs,
        "self_query": self_query_docs,
    }

    rows = []

    # ---------------------------------------------------------
    # Build one table row per retrieval technique
    # ---------------------------------------------------------
    for technique, docs in results.items():
        sources = {
            document_identity(doc)
            for doc in docs
        }

        categories = sorted({
            doc.metadata.get("category", "unknown")
            for doc in docs
        })

        match_rate = category_match_rate(
            docs=docs,
            expected_category=expected_category,
        )

        rows.append(
            {
                "technique": technique,
                "documents_retrieved": len(docs),
                "unique_sources": len(sources),
                "new_vs_baseline": len(
                    sources - baseline_sources
                ),
                "category_match_rate": match_rate,
                "categories": (
                    ", ".join(categories)
                    if categories
                    else "unknown"
                ),
            }
        )

    comparison_df = pd.DataFrame(rows)

    # ---------------------------------------------------------
    # Print transformation details
    # ---------------------------------------------------------
    print(
        f"\n{Fore.BLUE}{Style.BRIGHT}"
        "=== Query Transformation Details ==="
        f"{Style.RESET_ALL}"
    )

    print(
        f"{Fore.YELLOW}"
        "Original query:"
        f"{Style.RESET_ALL} "
        f"{question}"
    )

    print(
        f"{Fore.YELLOW}"
        "Rewritten query:"
        f"{Style.RESET_ALL} "
        f"{rewritten_query}"
    )

    print(
        f"{Fore.YELLOW}"
        "Step-back query:"
        f"{Style.RESET_ALL} "
        f"{step_back_query_text}"
    )

    print(
        f"{Fore.YELLOW}"
        "HyDE hypothetical document:"
        f"{Style.RESET_ALL}"
    )

    print(hypothetical_document)

    print(
        f"\n{Fore.YELLOW}"
        "Combined transformations:"
        f"{Style.RESET_ALL}"
    )

    print(transformations)

    # ---------------------------------------------------------
    # Print self-query interpretation
    # ---------------------------------------------------------
    print(
        f"\n{Fore.MAGENTA}{Style.BRIGHT}"
        "=== Self-Query Interpretation ==="
        f"{Style.RESET_ALL}"
    )

    print(
        f"{Fore.YELLOW}"
        "Semantic query:"
        f"{Style.RESET_ALL} "
        f"{structured_query.semantic_query}"
    )

    print(
        f"{Fore.YELLOW}"
        "Metadata category filter:"
        f"{Style.RESET_ALL} "
        f"{structured_query.category}"
    )

    # ---------------------------------------------------------
    # Print the comparison table
    # ---------------------------------------------------------
    print_colored_dataframe(
        dataframe=comparison_df,
        title="Retrieval Technique Comparison",
        color=Fore.CYAN,
    )

    return comparison_df


comparison_question = (
    "Show me Business articles about technology-company risks."
)

comparison_df = compare_retrieval_techniques(
    question=comparison_question,
    expected_category="Business",
    k=5,
)
        
print_colored_dataframe(
    dataframe=table,
    title="Retrieval Techniques Comparison",
)

### Interpreting the comparison

- Baseline retrieval is usually fastest because it requires no transformation call.
- Combined query transformation can improve recall, but it adds several LLM and retrieval calls.
- Self-querying adds one LLM parsing call but can substantially improve precision when the request contains a metadata constraint.
- The best method depends on the query. Self-querying provides little benefit when no useful metadata is available.

## 18. Create retrieval tools

In [ ]:
from langchain.tools import tool


@tool
def search_news_archive(query: str) -> str:
    """Search the historical AG News archive without context optimization."""

    docs = baseline_retriever.invoke(query)

    if not docs:
        return (
            "STATUS: NO_RESULTS\n"
            "No relevant articles were found."
        )

    context = format_documents(docs)

    return (
        "STATUS: RESULTS_FOUND\n"
        f"CONTEXT_TOKENS: {count_tokens(context)}\n\n"
        f"{context}"
    )


@tool
def search_optimized_news_archive(query: str) -> str:
    """Search AG News and optimize context automatically from token count."""

    docs = broad_retriever.invoke(query)

    if not docs:
        return (
            "STATUS: NO_RESULTS\n"
            "No relevant articles were found."
        )

    context, report = optimize_context(
        query=query,
        docs=docs,
    )

    return (
        "STATUS: RESULTS_FOUND\n"
        f"OPTIMIZATION_ACTION: {report['action']}\n"
        f"TOKENS_BEFORE: {report['before_tokens']}\n"
        f"TOKENS_AFTER: {report['after_tokens']}\n"
        f"TOKEN_REDUCTION: {report['token_reduction']}\n\n"
        f"{context}"
    )


@tool
def search_news_archive_transformed(query: str) -> str:
    """Search AG News with rewriting, expansion, step-back prompting, and HyDE."""

    docs, transformation_report = retrieve_with_all_transformations(
        question=query
    )

    if not docs:
        return (
            "STATUS: NO_RESULTS\n"
            "No relevant articles were found."
        )

    context, optimization_report = optimize_context(
        query=query,
        docs=docs,
    )

    return (
        "STATUS: RESULTS_FOUND\n"
        f"REWRITTEN_QUERY: {transformation_report['rewritten_query']}\n"
        f"EXPANDED_QUERIES: {transformation_report['expanded_queries']}\n"
        f"STEP_BACK_QUERY: {transformation_report['step_back_query']}\n"
        f"DOCUMENTS_RETRIEVED: {transformation_report['documents_retrieved']}\n"
        f"OPTIMIZATION_ACTION: {optimization_report['action']}\n"
        f"TOKENS_BEFORE: {optimization_report['before_tokens']}\n"
        f"TOKENS_AFTER: {optimization_report['after_tokens']}\n\n"
        f"{context}"
    )
@tool
def search_self_query_news_archive(query: str) -> str:
    """Search AG News using semantic search plus an inferred category filter."""

    docs, structured_query = self_query_retrieve(
        question=query,
        k=5,
    )

    if not docs:
        return (
            "STATUS: NO_RESULTS\n"
            "No relevant articles matched the semantic query "
            "and metadata filter."
        )

    context, optimization_report = optimize_context(
        query=structured_query.semantic_query,
        docs=docs,
    )

    return (
        "STATUS: RESULTS_FOUND\n"
        f"SEMANTIC_QUERY: {structured_query.semantic_query}\n"
        f"CATEGORY_FILTER: {structured_query.category}\n"
        f"DOCUMENTS_RETRIEVED: {len(docs)}\n"
        f"OPTIMIZATION_ACTION: {optimization_report['action']}\n"
        f"TOKENS_BEFORE: {optimization_report['before_tokens']}\n"
        f"TOKENS_AFTER: {optimization_report['after_tokens']}\n\n"
        f"{context}"
    )

### CRAG retrieval tool

The CRAG retriever is exposed as a LangChain tool so an agent can invoke the same corrective workflow.

In [ ]:
@tool
def search_crag_news_archive(query: str) -> str:
    """
    Search AG News with corrective retrieval.

    Initial documents are graded for relevance. When the evidence is weak,
    the query is rewritten and a broader corrective retrieval is performed.
    """
    docs, crag_report = crag_retrieve(
        question=query,
        k=5,
        minimum_relevant_docs=2,
    )

    if not docs:
        return (
            "STATUS: NO_RESULTS\n"
            "No relevant articles were found after corrective retrieval."
        )

    context, optimization_report = optimize_context(
        query=query,
        docs=docs,
    )

    return (
        "STATUS: RESULTS_FOUND\n"
        f"RETRIEVAL_PATH: {crag_report['retrieval_path']}\n"
        f"INITIAL_DOCUMENTS: {crag_report['initial_documents']}\n"
        f"INITIAL_RELEVANT_DOCUMENTS: "
        f"{crag_report['initial_relevant_documents']}\n"
        f"INITIAL_RELEVANCE_RATE: "
        f"{crag_report['initial_relevance_rate']:.0%}\n"
        f"CORRECTION_REQUIRED: "
        f"{crag_report['correction_required']}\n"
        f"CORRECTED_QUERY: "
        f"{crag_report['corrected_query'] or 'NOT_REQUIRED'}\n"
        f"FINAL_DOCUMENTS: {crag_report['final_documents']}\n"
        f"OPTIMIZATION_ACTION: "
        f"{optimization_report['action']}\n"
        f"TOKENS_BEFORE: "
        f"{optimization_report['before_tokens']}\n"
        f"TOKENS_AFTER: "
        f"{optimization_report['after_tokens']}\n\n"
        f"{context}"
    )


## 19. Create agents

In [ ]:
from langchain.agents import create_agent

baseline_agent = create_agent(
    model=model,
    tools=[search_news_archive],
    system_prompt=GROUNDED_PROMPT,
)

optimized_agent = create_agent(
    model=model,
    tools=[search_optimized_news_archive],
    system_prompt=GROUNDED_PROMPT,
)

transformed_agent = create_agent(
    model=model,
    tools=[search_news_archive_transformed],
    system_prompt=GROUNDED_PROMPT,
)

prompt_agents = {
    "Weak": create_agent(
        model=model,
        tools=[search_news_archive],
        system_prompt=WEAK_PROMPT,
    ),
    "Grounded": baseline_agent,
    "Beginner": create_agent(
        model=model,
        tools=[search_news_archive],
        system_prompt=BEGINNER_FRIENDLY_PROMPT,
    ),
    "Executive": create_agent(
        model=model,
        tools=[search_news_archive],
        system_prompt=EXECUTIVE_PROMPT,
    ),
    "Clarifying": create_agent(
        model=model,
        tools=[search_news_archive],
        system_prompt=CLARIFYING_PROMPT,
    ),
    "Analytical": create_agent(
        model=model,
        tools=[search_news_archive],
        system_prompt=ANALYTICAL_PROMPT,
    ),
}

print("Agents created.")
self_query_agent = create_agent(
    model=model,
    tools=[search_self_query_news_archive],
    system_prompt=GROUNDED_PROMPT,
)

print("Self-query agent created.")

### CRAG agent

This agent uses the same grounded system prompt as the other agents, but its tool validates and conditionally corrects retrieval before returning evidence.

In [ ]:
crag_agent = create_agent(
    model=model,
    tools=[search_crag_news_archive],
    system_prompt=GROUNDED_PROMPT,
)

print("CRAG agent created.")


## 20. Invoke agents and print logs

In [ ]:
import time


def ask_agent(
    agent,
    question: str,
    label: str,
) -> dict:
    print("\n" + "=" * 80)
    print(f"{Fore.CYAN}{label}{Style.RESET_ALL}")
    print("=" * 80)
    print(f"{Fore.YELLOW}[Question]{Style.RESET_ALL}")
    print(question)

    started = time.perf_counter()

    result = agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": question,
                }
            ]
        }
    )

    latency = time.perf_counter() - started
    answer = result["messages"][-1].content

    output = {
        "pipeline": label,
        "latency_seconds": round(latency, 3),
        "answer_tokens": count_tokens(answer),
        "answer": answer,
    }

    print(f"\n{Fore.YELLOW}[Latency]{Style.RESET_ALL}")
    print(f"{output['latency_seconds']} seconds")

    print(f"\n{Fore.YELLOW}[Answer tokens]{Style.RESET_ALL}")
    print(output["answer_tokens"])

    print(f"\n{Fore.GREEN}[Answer]{Style.RESET_ALL}")
    print(answer)

    return output

### Baseline, optimized, and transformed comparison

In [ ]:
from colorama import Fore, Style


def print_colored_dataframe(
    dataframe: pd.DataFrame,
    title: str,
    color: str = Fore.CYAN,
):
    """
    Pretty-print a DataFrame with a colored Category Match Rate column.
    """

    print(
        f"\n{color}{Style.BRIGHT}"
        f"{'=' * 90}\n"
        f"{title.center(90)}\n"
        f"{'=' * 90}"
        f"{Style.RESET_ALL}"
    )

    headers = [
        "Technique",
        "Docs",
        "Sources",
        "New",
        "Category Match",
        "Categories",
    ]

    print(
        f"{Style.BRIGHT}"
        f"{headers[0]:<15}"
        f"{headers[1]:>8}"
        f"{headers[2]:>10}"
        f"{headers[3]:>8}"
        f"{headers[4]:>20}"
        f"   {headers[5]}"
        f"{Style.RESET_ALL}"
    )

    print("-" * 90)

    for _, row in dataframe.iterrows():

        rate = row.get("category_match_rate", None)

        if rate is None:
            rate_text = "-"
            rate_color = Fore.WHITE

        else:
            percentage = rate * 100

            if percentage == 100:
                rate_color = Fore.GREEN

            elif percentage >= 80:
                rate_color = Fore.CYAN

            elif percentage >= 60:
                rate_color = Fore.YELLOW

            else:
                rate_color = Fore.RED

            rate_text = f"{percentage:.0f}%"

        print(
            f"{row['technique']:<15}"
            f"{row['documents_retrieved']:>8}"
            f"{row['unique_sources']:>10}"
            f"{row['new_vs_baseline']:>8}"
            f"{rate_color}{rate_text:>20}{Style.RESET_ALL}"
            f"   {row.get('categories', 'unknown')}"
        )
        
print_colored_dataframe(
    dataframe=comparison_df,
    title="Retrieval Techniques Comparison",
)

### CRAG agent demonstration

In [ ]:
ask_agent(
    agent=crag_agent,
    question=(
        "What business risks do technology companies face?"
    ),
    label="CRAG Agent",
)


### Prompt-variant comparison

In [ ]:
prompt_question = "What do the articles say about technology companies?"

for name, agent in prompt_agents.items():
    ask_agent(
        agent=agent,
        question=prompt_question,
        label=name,
    )

## 21. Ambiguity and insufficient-context tests

In [ ]:
ask_agent(
    agent=prompt_agents["Clarifying"],
    question="What happened with Apple?",
    label="Clarification Test",
)

ask_agent(
    agent=baseline_agent,
    question="What did Apple announce yesterday?",
    label="Insufficient Context Test",
)